# Deploy Credit Score Model ke SageMaker Endpoint
Jalankan notebook ini SETELAH `pipeline_aws.py` selesai dan `best_model.pkl` sudah ada.

Urutan: Package model → Upload S3 → Deploy Endpoint → Smoke test


In [2]:
!pip install --upgrade pip setuptools wheel

In [5]:
!pip install scikit-learn==1.4.2 --only-binary=:all:

  Using cached scikit_learn-1.4.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 185.0 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [20]:
import boto3
import sagemaker

print("boto3 & sagemaker ready")

boto3 & sagemaker ready


In [21]:
# Cek role yang dipakai
role = sagemaker.get_execution_role()
print(role)

arn:aws:iam::484191005961:role/LabRole


In [22]:
# Cek bucket yang tersedia
s3 = boto3.client("s3")
response = s3.list_buckets()
for bucket in response['Buckets']:
    print(bucket['Name'])

sagemaker-us-east-1-484191005961


## Step 1 — Package model jadi .tar.gz
SageMaker Endpoint WAJIB terima model dalam bentuk archive .tar.gz, bukan .pkl polos.


In [23]:
import tarfile
import os

# Sesuaikan path ini dengan lokasi best_model.pkl hasil pipeline_aws.py
MODEL_PKL_PATH = "/home/ec2-user/SageMaker/credit_scoring/model/best_model.pkl"

os.makedirs("model_package", exist_ok=True)

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add(MODEL_PKL_PATH, arcname="best_model.pkl")

print("✅ model.tar.gz berhasil dibuat")

✅ model.tar.gz berhasil dibuat


## Step 2 — Upload model ke S3
Deployment akan ambil model dari S3, bukan dari lokal.


In [24]:
s3 = boto3.client("s3")

# ---- EDIT INI: ganti dengan nama bucket kamu ----
BUCKET = "sagemaker-us-east-1-484191005961"
# --------------------------------------------------

s3.upload_file(
    "model.tar.gz",
    BUCKET,
    "model/model.tar.gz"
)
print(f"✅ Model diupload ke s3://{BUCKET}/model/model.tar.gz")

✅ Model diupload ke s3://sagemaker-us-east-1-484191005961/model/model.tar.gz


In [25]:
import tarfile

# Extract model.tar.gz ke folder model_package/ supaya bisa dites lewat model_fn()
with tarfile.open("model.tar.gz", "r:gz") as tar:
    tar.extractall("model_package")

print("✅ model.tar.gz berhasil di-extract ke model_package/")
print("Isi model_package/:", os.listdir("model_package"))

✅ model.tar.gz berhasil di-extract ke model_package/
Isi model_package/: ['best_model.pkl']


## Step 3 — Cek inference.py bisa jalan (opsional tapi disarankan)


In [26]:
from inference_aws import model_fn

model = model_fn("model_package")
print("✅ Model berhasil di-load lewat model_fn")

✅ Model (Unified Pipeline) loaded from model_package/best_model.pkl
✅ Model berhasil di-load lewat model_fn


## Step 4 — Deploy ke SageMaker Endpoint
Proses ini memakan waktu 5-8 menit.


In [28]:
import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel

# ---- EDIT INI ----
BUCKET = "sagemaker-us-east-1-484191005961"
MODEL_S3_KEY = "model/model.tar.gz"
ENDPOINT_NAME = "credit-score-endpoint1"
# -------------------

REGION = "us-east-1"
INSTANCE_TYPE = "ml.m5.large"
FRAMEWORK_VERSION = "1.4-2"


def get_lab_role_arn() -> str:
    iam = boto3.client("iam")
    return iam.get_role(RoleName="LabRole")["Role"]["Arn"]


def main():
    boto3.setup_default_session(region_name=REGION)
    sm_session = sagemaker.Session()
    role_arn = get_lab_role_arn()
    model_s3_uri = f"s3://{BUCKET}/{MODEL_S3_KEY}"

    print(f"Role:      {role_arn}")
    print(f"Model URI: {model_s3_uri}")
    print(f"Endpoint:  {ENDPOINT_NAME}")

    model = SKLearnModel(
        model_data=model_s3_uri,
        role=role_arn,
        entry_point="inference_aws.py",   # file handler model_fn/input_fn/dll
        source_dir=".",
        framework_version=FRAMEWORK_VERSION,
        sagemaker_session=sm_session,
    )

    print("\nDeploying endpoint (5-8 menit)...")
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=INSTANCE_TYPE,
        endpoint_name=ENDPOINT_NAME
    )

    print("\n✅ Endpoint berhasil deploy!")
    return predictor

predictor = main()

Role:      arn:aws:iam::484191005961:role/LabRole
Model URI: s3://sagemaker-us-east-1-484191005961/model/model.tar.gz
Endpoint:  credit-score-endpoint1

Deploying endpoint (5-8 menit)...
-----!
✅ Endpoint berhasil deploy!


## Step 5 — Smoke Test
Kirim 1 contoh data untuk memastikan endpoint benar-benar merespons.


In [19]:
import json

REGION = "us-east-1"
ENDPOINT_NAME = "credit-score-endpoint"

# Contoh 1 baris data (21 fitur, urutan sesuai FEATURE_NAMES di inference_aws.py)
sample = {
    "instances": [[
        29, 85583.02, 7366.92, 4, 1, 2, 3, 14, 50, 9.51, 2,
        44.08, 37.49, 222, 184.68, 581.66, 240.35,
        7, 0, 0, 1   # encoded: Occupation, Credit_Mix, Payment_of_Min_Amount, Payment_Behaviour
    ]]
}

runtime = boto3.client("sagemaker-runtime", region_name=REGION)
response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps(sample),
)

print("Smoke test response:")
print(response["Body"].read().decode("utf-8"))

Smoke test response:
{"probabilities": [[0.5597417253209276, 0.2613334261031023, 0.17892484857597005]], "predictions": [0], "labels": ["Good"]}


## Cek Log kalau Ada Error
Buka CloudWatch Logs: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#logsV2:log-groups


## Cleanup — Hapus Endpoint (WAJIB setelah selesai testing!)
Endpoint yang menyala terus akan kena biaya. Hapus setelah screenshot bukti selesai diambil.


In [15]:
# Hapus endpoint setelah selesai supaya tidak kena biaya terus-menerus
predictor.delete_endpoint()
print("✅ Endpoint dihapus")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 # Hapus endpoint setelah selesai supaya tidak kena biaya terus-menerus                       │
│ ❱ 2 predictor.delete_endpoint()                                                                  │
│   3 print("✅ Endpoint dihapus")                                                                 │
│   4                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'predictor' is not defined